### MFR Demo

In [3]:
import pandas as pd
from datasets import load_dataset
from utils import counting, mfr, evaluate

# Load data
dev_pub_data = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Select data
lang = "en"
train, val = dev_pub_data["train"], dev_pub_data["validation"]

# Smoke test baseline
counts = counting(train)
out = mfr(['bcause', 'u', 'r', 'funny'], counts)
print("Smoke test:", out)

## Inference
ds = pd.DataFrame(val)
ds['pred'] = ds['raw'].apply(lambda x: mfr(x, counts))

## Evaluate 
evaluate(ds['raw'].tolist(), ds['norm'].tolist(), ds['pred'].tolist())

Smoke test: ['because', 'u', 'rồi', 'funny']
Baseline acc.(LAI): 88.48
Accuracy:           92.97
ERR:                39.02


(0.8847954569019836, 0.9297478803927355, 0.3901966214345056)

### Submission demo

In [ ]:
data, save_path = load_dataset("weerayut/multilexnorm2026-dev-pub"), "outputs/submission_dev"
#data, save_path = load_dataset("weerayut/multilexnorm2026-full-pub"), "outputs/submission_full"

In [ ]:
def prediction(train, test):
    train_df = train.to_pandas()
    test_df  = test.to_pandas()

    # Build a per-language dictionary
    count_langs = {} 
    for lang in train_df["lang"].unique():
        train_lang = train_df.loc[train_df["lang"] == lang]
        count_langs[lang] = counting(train_lang.to_dict(orient="records"))

    # Prediction per language
    test_df["pred"] = test_df.apply(
        lambda r: mfr(r["raw"], count_langs.get(r["lang"])),
        axis=1
    )
    return test_df

In [ ]:
from datasets import concatenate_datasets

## predict and save dev
train = concatenate_datasets([data['train'], data['validation']])
out = prediction(train, data['test'])
out.to_json(f"{save_path}/predictions.json", orient="records")

In [ ]:
out[['raw', 'pred', 'lang']].head(3)

### Zip files

In [ ]:
from utils import zip_files_flat

zip_files_flat(save_path, f"{save_path}.zip")

#Hugging face

## ByT5 Training + Submission

Run the cells in this section top-to-bottom to fine-tune ByT5 and build the submission zip.

**Where to run this**: the training cell needs a GPU. On vast.ai, connect VS Code via
Remote-SSH and reopen this notebook from inside the cloned repo before running. On the
3090 vast machine dev-pub finishes in about 4 hours; on Mac MPS it takes 20+ hours.

**Flow**: training writes the best checkpoint to `./checkpoints/byt5-base-baseline/final/`, then
the loader cell mounts it and the prediction cells reuse the same submission pipeline as
the MFR baseline (predictions.json -> zip).

**Model**: single byt5-base (~580M params) fine-tuned on all 17 languages at once.
No per-language oversampling — uniform sampling across languages worked best in our
ablation (see the "ko oversample ablation" section near the bottom of this notebook).

**Output**: predictions in `outputs/submission_dev_byt5_baseline/`. To boost ja and other
high-LAI languages further, apply the Copy Fallback post-processing in the next section.

In [1]:
# === ByT5 training utilities (replaces train_byt5.py) ===
# Defines the train_byt5(...) entry point used by all the training cells below.

import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, load_dataset
from torch.utils.data import WeightedRandomSampler
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5ForConditionalGeneration,
)


def _set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _build_word_pairs(split):
    langs, raws, norms = [], [], []
    for item in split:
        lang = item["lang"]
        for r, n in zip(item["raw"], item["norm"]):
            langs.append(lang)
            raws.append(r)
            norms.append(n)
    return Dataset.from_dict({"lang": langs, "raw": raws, "norm": norms})


def _make_tokenize_fn(tokenizer, max_input_len, max_target_len):
    def fn(batch):
        inputs = [f"{l}: {r}" for l, r in zip(batch["lang"], batch["raw"])]
        targets = batch["norm"]
        model_inputs = tokenizer(inputs, max_length=max_input_len, truncation=True, padding=False)
        labels = tokenizer(text_target=targets, max_length=max_target_len, truncation=True, padding=False)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return fn


def _detect_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def _build_per_lang_subset(pairs, n_per_lang, seed):
    rng = np.random.default_rng(seed)
    by_lang = {}
    for i, lang in enumerate(pairs["lang"]):
        by_lang.setdefault(lang, []).append(i)
    chosen = []
    for lang, idxs in by_lang.items():
        if len(idxs) > n_per_lang:
            picked = rng.choice(idxs, size=n_per_lang, replace=False)
            chosen.extend(int(x) for x in picked)
        else:
            chosen.extend(idxs)
    chosen.sort()
    return pairs.select(chosen)


def _word_err(raws, golds, preds):
    n = len(raws)
    if n == 0:
        return 0.0, 0.0, 0.0
    correct_sys = sum(1 for p, g in zip(preds, golds) if p == g)
    correct_lai = sum(1 for r, g in zip(raws, golds) if r == g)
    acc_sys = correct_sys / n
    acc_lai = correct_lai / n
    denom = 1.0 - acc_lai
    err = (acc_sys - acc_lai) / denom if denom > 0 else 0.0
    return acc_lai, acc_sys, err


class _WeightedSamplerTrainer(Seq2SeqTrainer):
    def __init__(self, *args, train_sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_sample_weights = train_sample_weights

    def _get_train_sampler(self, *args, **kwargs):
        if self.train_sample_weights is None:
            return super()._get_train_sampler(*args, **kwargs)
        g = torch.Generator()
        g.manual_seed(self.args.seed)
        return WeightedRandomSampler(
            weights=self.train_sample_weights,
            num_samples=len(self.train_dataset),
            replacement=True,
            generator=g,
        )


def train_byt5(
    model_name="google/byt5-base",
    dataset_name="weerayut/multilexnorm2026-dev-pub",
    output_dir="./checkpoints/byt5-base-unified",
    epochs=5,
    batch_size=64,
    eval_batch_size=128,
    grad_accum=1,
    lr=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    max_input_len=64,
    max_target_len=64,
    seed=42,
    fp16=False,
    bf16=None,
    num_workers=None,
    early_stop_patience=2,
    save_total_limit=2,
    resume=False,
    oversample_weights=None,
    target_langs=None,
    eval_per_lang_n=200,
    gradient_checkpointing=False,
):
    """Fine-tune ByT5 on MultiLexNorm 2026. Outputs best model to output_dir/final/."""
    device = _detect_device()
    if bf16 is None:
        bf16 = device == "cuda"
    if num_workers is None:
        num_workers = 4 if device == "cuda" else 0
    print(f"[device] {device} (fp16={fp16}, bf16={bf16}, num_workers={num_workers})")

    _set_seed(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"[load] dataset = {dataset_name}")
    ds = load_dataset(dataset_name)
    print(f"[load] splits = {list(ds.keys())}")
    print(f"[load] train = {len(ds['train'])}  val = {len(ds['validation'])}")

    if target_langs:
        targets = set(target_langs)
        ds = ds.filter(lambda x: x["lang"] in targets)
        print(f"[filter] keep langs={sorted(targets)} -> train={len(ds['train'])} val={len(ds['validation'])}")

    print("[prep] flattening sentence -> word pairs ...")
    train_pairs = _build_word_pairs(ds["train"])
    val_pairs = _build_word_pairs(ds["validation"])
    print(f"[prep] train pairs = {len(train_pairs)}  val pairs = {len(val_pairs)}")

    train_counts = Counter(train_pairs["lang"])
    print(f"[stats] train pairs by lang: {dict(sorted(train_counts.items()))}")

    sample_weights = None
    if oversample_weights:
        sample_weights = [oversample_weights.get(l, 1.0) for l in train_pairs["lang"]]
        per_lang_w = Counter()
        for l, w in zip(train_pairs["lang"], sample_weights):
            per_lang_w[l] += w
        total_w = sum(sample_weights)
        ratios = {l: round(w / total_w * 100, 2) for l, w in sorted(per_lang_w.items())}
        print(f"[oversample] weights override: {oversample_weights}")
        print(f"[oversample] expected per-lang sampling fraction (%): {ratios}")

    print(f"[load] tokenizer + model = {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name, use_safetensors=True)

    tokenize_fn = _make_tokenize_fn(tokenizer, max_input_len, max_target_len)
    train_tok = train_pairs.map(
        tokenize_fn, batched=True, remove_columns=train_pairs.column_names, desc="tokenize train",
    )

    print(f"[eval] building per-lang val subset (n_per_lang={eval_per_lang_n})")
    eval_subset = _build_per_lang_subset(val_pairs, eval_per_lang_n, seed)
    eval_counts = Counter(eval_subset["lang"])
    print(f"[eval] subset size = {len(eval_subset)}  per-lang = {dict(sorted(eval_counts.items()))}")
    eval_langs_arr = list(eval_subset["lang"])
    eval_raws_arr = list(eval_subset["raw"])
    eval_norms_arr = list(eval_subset["norm"])
    eval_tok = eval_subset.map(
        tokenize_fn, batched=True, remove_columns=eval_subset.column_names, desc="tokenize eval_subset",
    )

    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

    def compute_metrics(eval_preds):
        preds = eval_preds.predictions
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.asarray(preds)
        if (preds == -100).any():
            preds = np.where(preds == -100, tokenizer.pad_token_id, preds)
        decoded = tokenizer.batch_decode(preds, skip_special_tokens=True)
        m = min(len(decoded), len(eval_langs_arr))
        by_lang = {}
        all_r, all_n, all_p = [], [], []
        for i in range(m):
            lang = eval_langs_arr[i]
            slot = by_lang.setdefault(lang, ([], [], []))
            slot[0].append(eval_raws_arr[i])
            slot[1].append(eval_norms_arr[i])
            slot[2].append(decoded[i])
            all_r.append(eval_raws_arr[i])
            all_n.append(eval_norms_arr[i])
            all_p.append(decoded[i])
        out = {}
        for lang in sorted(by_lang):
            r, n, p = by_lang[lang]
            _, _, err = _word_err(r, n, p)
            out[f"err_{lang}"] = round(err * 100, 4)
        _, _, err_avg = _word_err(all_r, all_n, all_p)
        out["err_average"] = round(err_avg * 100, 4)
        return out

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=save_total_limit,
        load_best_model_at_end=True,
        metric_for_best_model="err_average",
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=max_target_len,
        generation_num_beams=1,
        fp16=fp16,
        bf16=bf16 and not fp16,
        dataloader_num_workers=num_workers,
        report_to=["none"],
        seed=seed,
        gradient_checkpointing=gradient_checkpointing,
        gradient_checkpointing_kwargs={"use_reentrant": False} if gradient_checkpointing else None,
    )

    trainer = _WeightedSamplerTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stop_patience)],
        train_sample_weights=sample_weights,
    )

    resume_flag = False
    if resume:
        ckpts = sorted(output_dir.glob("checkpoint-*"))
        if ckpts:
            print(f"[resume] found {len(ckpts)} checkpoint(s); latest = {ckpts[-1].name}")
            resume_flag = True
        else:
            print("[resume] no checkpoint found, starting from scratch")
    print(f"[train] starting ... (resume={resume_flag})")
    trainer.train(resume_from_checkpoint=resume_flag)

    final_dir = output_dir / "final"
    print(f"[save] best model -> {final_dir}")
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    # Free disk: best checkpoint is now in final/, drop intermediate checkpoint-* dirs.
    import shutil
    for d in output_dir.iterdir():
        if d.is_dir() and d.name.startswith("checkpoint-"):
            shutil.rmtree(d)
            print(f"[cleanup] removed {d}")
    print("[done]")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Train UNIFIED multilingual ByT5-base on all 17 languages (skip if checkpoint exists).
# This is the WINNING config from our ablation study:
#   - byt5-base (not byt5-small): larger model gives ~+2 weighted ERR
#   - NO per-language oversampling: uniform sampling beats ko=3x oversample (see ablation
#     section at the bottom of the notebook for the negative result)
#   - 5 epochs, batch 64, lr 5e-4, default early stopping
#
# num_workers=0: vast.ai container blocks pidfd_getfd, so DataLoader workers
# can't share CUDA tensors across processes. Single-process loading is fine
# here because tokenization is pre-computed in .map().

import os

UNIFIED_OUT       = "./checkpoints/byt5-base-baseline"
UNIFIED_EPOCHS    = 5
UNIFIED_BATCH     = 64

if os.path.exists(f"{UNIFIED_OUT}/final/config.json"):
    print(f"[skip] checkpoint exists at {UNIFIED_OUT}/final")
else:
    train_byt5(
        model_name="google/byt5-base",
        output_dir=UNIFIED_OUT,
        epochs=UNIFIED_EPOCHS,
        batch_size=UNIFIED_BATCH,
        # no oversample_weights -> uniform sampling across all 17 langs
        num_workers=0,
        save_total_limit=1,
    )

In [ ]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

CKPT = "./checkpoints/byt5-base-baseline/final"
device = "cuda" if torch.cuda.is_available() else "cpu"

byt5_tok = AutoTokenizer.from_pretrained(CKPT)
byt5_model = T5ForConditionalGeneration.from_pretrained(CKPT).to(device).eval()
print(f"loaded {CKPT} on {device}")


def byt5_predict_words(words, langs, batch_size=256, max_input=64, max_target=64, num_beams=1):
    """Batched word-level prediction. Same shape as the MFR helper:
    words/langs are flat lists, returns a flat list of predicted words."""
    preds = []
    for i in range(0, len(words), batch_size):
        chunk_w = words[i:i + batch_size]
        chunk_l = langs[i:i + batch_size]
        inputs = [f"{l}: {w}" for l, w in zip(chunk_l, chunk_w)]
        enc = byt5_tok(inputs, return_tensors="pt", padding=True,
                       truncation=True, max_length=max_input).to(device)
        with torch.no_grad():
            out = byt5_model.generate(
                **enc, max_new_tokens=max_target, num_beams=num_beams,
            )
        preds.extend(byt5_tok.batch_decode(out, skip_special_tokens=True))
    return preds

In [4]:
# Sanity check on validation set: per-language ERR with the trained ByT5

byt5_data = load_dataset("weerayut/multilexnorm2026-dev-pub")

val_df = byt5_data["validation"].to_pandas()
flat_words, flat_langs, sent_lens = [], [], []
for _, row in val_df.iterrows():
    flat_words.extend(row["raw"])
    flat_langs.extend([row["lang"]] * len(row["raw"]))
    sent_lens.append(len(row["raw"]))

flat_preds = byt5_predict_words(flat_words, flat_langs)

val_preds, cursor = [], 0
for n in sent_lens:
    val_preds.append(flat_preds[cursor:cursor + n])
    cursor += n
val_df["pred"] = val_preds

evaluate(val_df["raw"].tolist(), val_df["norm"].tolist(), val_df["pred"].tolist())

KeyboardInterrupt: 

In [ ]:
# Predict on the official test split and save predictions.json (ByT5)

byt5_save_path = "outputs/submission_dev_byt5_baseline"
# byt5_save_path = "outputs/submission_full_byt5_baseline"  # for the final phase

test_df = byt5_data["test"].to_pandas()
flat_words, flat_langs, sent_lens = [], [], []
for _, row in test_df.iterrows():
    flat_words.extend(row["raw"])
    flat_langs.extend([row["lang"]] * len(row["raw"]))
    sent_lens.append(len(row["raw"]))

flat_preds = byt5_predict_words(flat_words, flat_langs)

test_preds, cursor = [], 0
for n in sent_lens:
    test_preds.append(flat_preds[cursor:cursor + n])
    cursor += n
test_df["pred"] = test_preds

Path(byt5_save_path).mkdir(parents=True, exist_ok=True)
test_df.to_json(f"{byt5_save_path}/predictions.json", orient="records")
test_df[["raw", "pred", "lang"]].head(3)

In [6]:
from utils import zip_files_flat
zip_files_flat(byt5_save_path, f"{byt5_save_path}.zip")

Created zip file: outputs/submission_dev_byt5.zip


## Copy Fallback Post-processing

The ByT5 model sometimes over-changes words that should have been left as-is.
This is especially costly for high-LAI languages (ja, ko, en, th) where the
ERR metric heavily penalises wrong edits.

We add a **deterministic post-processing step** that uses training statistics
to veto suspicious model edits:

```
for each (raw, model_pred):
    if model_pred == raw:                    # model didn't change -> keep
        keep model_pred
    elif raw is in train AND train shows raw is usually kept (>= threshold):
        veto the model -> return raw
    else:
        keep model_pred
```

This is **not a per-language module** — same logic applies to every language;
the per-language `threshold` is just a hyperparameter tuned on validation.

**No data leakage**:
  - thresholds are picked on val using TRAIN-ONLY counts
  - at test time, we can use TRAIN+VAL counts since test is disjoint from both

Result: weighted ERR 50.84 → **51.90** on dev-phase test (codabench).
The gain is concentrated in ja (7.91 → 16.78), which has the highest LAI (93.74%).

In [ ]:
# Build per-language raw -> {norm: count} table from train + validation.
# At test time this is safe (no leakage) because test is disjoint from both splits.

from collections import defaultdict


def build_counts(items):
    counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    for item in items:
        lang = item["lang"]
        for r, n in zip(item["raw"], item["norm"]):
            counts[lang][r][n] += 1
    return {lang: {raw: dict(d) for raw, d in c.items()} for lang, c in counts.items()}


# byt5_data is loaded by cell 14; reuse it
fallback_counts = build_counts(list(byt5_data["train"]) + list(byt5_data["validation"]))
print({lang: len(fallback_counts[lang]) for lang in sorted(fallback_counts)})


In [ ]:
# Per-language copy-fallback configs from HONEST val sweep (train-only counts).
# Each entry: (min_count, threshold).
# These were tuned on the byt5-base-baseline checkpoint's val predictions.
# Languages absent from val (da, de, es, id, iden, it, nl, tr, trde) are skipped
# (no val signal to tune on -> safer to not apply fallback there).

FALLBACK_CFG = {
    "ko": (1, 0.60),
    "ja": (1, 0.40),
    "th": (1, 0.65),
    "sr": (1, 0.30),
    "en": (1, 0.55),
    "hr": (1, 0.50),
    "sl": (1, 0.50),
    "vi": (1, 0.40),
}


def apply_fallback(raw, pred, lang, counts=fallback_counts, cfg=FALLBACK_CFG):
    if pred == raw:
        return pred                       # model didn't change -> nothing to veto
    if lang not in cfg:
        return pred                       # no fallback for this lang
    min_count, threshold = cfg[lang]
    lang_c = counts.get(lang, {})
    if raw not in lang_c:
        return pred                       # unseen raw -> trust model
    total = sum(lang_c[raw].values())
    if total < min_count:
        return pred
    keep_count = lang_c[raw].get(raw, 0)
    return raw if keep_count / total >= threshold else pred


# quick smoke test
print("smoke:", apply_fallback("the", "thee", "en"))   # changes only fire when keep_ratio high


In [ ]:
# Apply copy fallback to the test predictions saved by cell 15, then zip.

import json
import zipfile
from pathlib import Path

raw_pred_path = Path(byt5_save_path) / "predictions.json"
fb_save_path = byt5_save_path + "_fb"          # e.g. submission_dev_byt5_baseline_fb
fb_zip_path = fb_save_path + ".zip"

test_data = json.load(open(raw_pred_path))
n_modified = defaultdict(int)
for item in test_data:
    new_preds = []
    for r, p in zip(item["raw"], item["pred"]):
        np_ = apply_fallback(r, p, item["lang"])
        if np_ != p:
            n_modified[item["lang"]] += 1
        new_preds.append(np_)
    item["pred"] = new_preds

print("[fallback] words modified per language:")
for l in sorted(n_modified):
    print(f"  {l}: {n_modified[l]}")

Path(fb_save_path).mkdir(parents=True, exist_ok=True)
with open(f"{fb_save_path}/predictions.json", "w") as f:
    json.dump(test_data, f, ensure_ascii=False)
with zipfile.ZipFile(fb_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(f"{fb_save_path}/predictions.json", arcname="predictions.json")
print(f"[fallback] saved {fb_zip_path}")


## ByT5 Hybrid Submission (multilingual + ko/ja monolingual)

Korean and Japanese underperformed in the single multilingual model. To address
that, we train two dedicated monolingual models alongside the multilingual
byt5-base and dispatch each test sentence to the model for its language at
inference time.

Run the cells below in order: train Korean, train Japanese, predict + zip.

In [4]:
# Train multilingual ByT5-base on 15 languages (ko and ja are excluded —
# they get dedicated monolingual models below). Skip if checkpoint exists.

import os

BASE_OUT        = "./checkpoints/byt5-base"
BASE_EPOCHS     = 5
BASE_BATCH      = 64
BASE_GRAD_ACCUM = 1
BASE_LANGS      = ["da", "de", "en", "es", "hr", "id", "iden", "it",
                   "nl", "sl", "sr", "th", "tr", "trde", "vi"]

if os.path.exists(f"{BASE_OUT}/final/config.json"):
    print(f"[skip] checkpoint exists at {BASE_OUT}/final")
else:
    train_byt5(
        model_name="google/byt5-base",
        output_dir=BASE_OUT,
        target_langs=BASE_LANGS,
        epochs=BASE_EPOCHS,
        batch_size=BASE_BATCH,
        grad_accum=BASE_GRAD_ACCUM,
        save_total_limit=1,
    )


[device] cuda (fp16=False, bf16=True, num_workers=4)
[load] dataset = weerayut/multilexnorm2026-dev-pub


[load] splits = ['train', 'validation', 'test']
[load] train = 39178  val = 8408


Filter:   0%|          | 0/39178 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5972 [00:00<?, ? examples/s]

[filter] keep langs=['da', 'de', 'en', 'es', 'hr', 'id', 'iden', 'it', 'nl', 'sl', 'sr', 'th', 'tr', 'trde', 'vi'] -> train=35345 val=7891
[prep] flattening sentence -> word pairs ...
[prep] train pairs = 565951  val pairs = 112578


KeyboardInterrupt: 

In [ ]:
# Train Korean-only ByT5-base (skip if checkpoint exists).

import os

KO_OUT          = "./checkpoints/byt5-ko"
KO_EPOCHS       = 20
KO_BATCH        = 32
KO_GRAD_ACCUM   = 2
KO_EARLY_STOP   = 4

if os.path.exists(f"{KO_OUT}/final/config.json"):
    print(f"[skip] checkpoint exists at {KO_OUT}/final")
else:
    train_byt5(
        model_name="google/byt5-base",
        output_dir=KO_OUT,
        target_langs=["ko"],
        epochs=KO_EPOCHS,
        batch_size=KO_BATCH,
        grad_accum=KO_GRAD_ACCUM,
        early_stop_patience=KO_EARLY_STOP,
        save_total_limit=1,
    )


In [ ]:
# Train Japanese-only ByT5-base (skip if checkpoint exists).

import os

JA_OUT          = "./checkpoints/byt5-ja"
JA_EPOCHS       = 10
JA_BATCH        = 32
JA_GRAD_ACCUM   = 2
JA_EARLY_STOP   = 3

if os.path.exists(f"{JA_OUT}/final/config.json"):
    print(f"[skip] checkpoint exists at {JA_OUT}/final")
else:
    train_byt5(
        model_name="google/byt5-base",
        output_dir=JA_OUT,
        target_langs=["ja"],
        epochs=JA_EPOCHS,
        batch_size=JA_BATCH,
        grad_accum=JA_GRAD_ACCUM,
        early_stop_patience=JA_EARLY_STOP,
        save_total_limit=1,
    )


In [1]:
# Hybrid inference: byt5-base multilingual for 15 languages,
# byt5-ko / byt5-ja for Korean and Japanese.

from pathlib import Path
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from datasets import load_dataset

HYBRID_SAVE = "outputs/submission_dev_byt5_hybrid"
MULTI_CKPT = "./checkpoints/byt5-base/final"
LANG_CKPT = {
    "ko": "./checkpoints/byt5-ko/final",
    "ja": "./checkpoints/byt5-ja/final",
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

byt5_data = load_dataset("weerayut/multilexnorm2026-dev-pub")
test_df = byt5_data["test"].to_pandas()
test_df["pred"] = [list() for _ in range(len(test_df))]

def predict_rows(ckpt, indices, batch_size=128, max_input=64, max_target=64):
    if not indices:
        return
    flat_w, flat_l, sent_lens = [], [], []
    for i in indices:
        row = test_df.iloc[i]
        flat_w.extend(row["raw"])
        flat_l.extend([row["lang"]] * len(row["raw"]))
        sent_lens.append(len(row["raw"]))
    print(f"[{ckpt}] {len(flat_w)} words from {len(indices)} sentences")
    tok = AutoTokenizer.from_pretrained(ckpt)
    model = T5ForConditionalGeneration.from_pretrained(ckpt).to(device).eval()
    preds = []
    for s in range(0, len(flat_w), batch_size):
        cw, cl = flat_w[s:s + batch_size], flat_l[s:s + batch_size]
        enc = tok([f"{l}: {w}" for l, w in zip(cl, cw)],
                  return_tensors="pt", padding=True, truncation=True,
                  max_length=max_input).to(device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_target, num_beams=1)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    cursor = 0
    for n, i in zip(sent_lens, indices):
        test_df.at[i, "pred"] = preds[cursor:cursor + n]
        cursor += n
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

default_idx = [i for i in range(len(test_df)) if test_df.at[i, "lang"] not in LANG_CKPT]
lang_idx = {lang: [i for i in range(len(test_df)) if test_df.at[i, "lang"] == lang]
            for lang in LANG_CKPT}

print(f"default model -> {len(default_idx)} sentences")
for lang, idxs in lang_idx.items():
    print(f"{lang} model    -> {len(idxs)} sentences")

predict_rows(MULTI_CKPT, default_idx)
for lang in sorted(lang_idx):
    predict_rows(LANG_CKPT[lang], lang_idx[lang])

Path(HYBRID_SAVE).mkdir(parents=True, exist_ok=True)
test_df.to_json(f"{HYBRID_SAVE}/predictions.json", orient="records")
test_df[["raw", "pred", "lang"]].head(3)

device: cuda
default model -> 5561 sentences
ko model    -> 107 sentences
ja model    -> 304 sentences
[./checkpoints/byt5-base/final] 84109 words from 5561 sentences


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. 

[./checkpoints/byt5-ja/final] 10710 words from 304 sentences


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. 

[./checkpoints/byt5-ko/final] 823 words from 107 sentences


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. 

,raw,pred,lang
0,"[Jeg, skaelver, .]","[Jeg, skælver, .]",da
1,"[Jeg, stryger, en, taendstik, .]","[Jeg, stryger, en, tændstik, .]",da
2,"[Jeg, tager, den, vaek, igen, .]","[Jeg, tager, den, væk, igen, .]",da


In [2]:
from utils import zip_files_flat
zip_files_flat(HYBRID_SAVE, f"{HYBRID_SAVE}.zip")

Created zip file: outputs/submission_dev_byt5_hybrid.zip


## ByT5 Large for ko/ja

Tests whether scaling the two weakest languages (ko, ja) from byt5-base (~580M) to byt5-large (~1.2B) closes the gap. Multilingual model for the other 15 languages stays at byt5-base.

**Training strategy**: fixed-epoch training (no early stopping) with best-checkpoint selection via `load_best_model_at_end=True`. This matches the ÚFAL 2021 paper setup and avoids prematurely stopping on temporary validation plateaus that could miss a late improvement.

Notes:
- Effective batch kept at 64 (`batch_size=16`, `grad_accum=4`) so the optimization regime matches the base run.
- Learning rate dropped to `3e-4` (ÚFAL paper finding: large benefits from lower lr).
- `early_stop_patience=999` effectively disables the callback so every epoch is evaluated; the trainer still loads the best validation checkpoint at the end.

In [ ]:
# Train Korean-only ByT5-large (skip if checkpoint exists).
# Method 2: train all epochs, no early stopping. Best validation checkpoint is
# auto-loaded via load_best_model_at_end=True.
# OOM fix: batch_size=8 + grad_accum=8 (effective 64 kept), gradient checkpointing on.

import os

KO_LARGE_OUT        = "./checkpoints/byt5-ko-large"
KO_LARGE_EPOCHS     = 15
KO_LARGE_BATCH      = 8
KO_LARGE_GRAD_ACCUM = 8
KO_LARGE_LR         = 3e-4
KO_LARGE_EARLY_STOP = 999  # effectively disabled

if os.path.exists(f"{KO_LARGE_OUT}/final/config.json"):
    print(f"[skip] checkpoint exists at {KO_LARGE_OUT}/final")
else:
    train_byt5(
        model_name="google/byt5-large",
        output_dir=KO_LARGE_OUT,
        target_langs=["ko"],
        epochs=KO_LARGE_EPOCHS,
        batch_size=KO_LARGE_BATCH,
        grad_accum=KO_LARGE_GRAD_ACCUM,
        lr=KO_LARGE_LR,
        early_stop_patience=KO_LARGE_EARLY_STOP,
        save_total_limit=1,
        gradient_checkpointing=True,
    )

In [ ]:
# Train Japanese-only ByT5-large (skip if checkpoint exists).
# Method 2: train all epochs, no early stopping. Best validation checkpoint is
# auto-loaded via load_best_model_at_end=True.
# OOM fix: batch_size=8 + grad_accum=8 (effective 64 kept), gradient checkpointing on.

import os

JA_LARGE_OUT        = "./checkpoints/byt5-ja-large"
JA_LARGE_EPOCHS     = 10
JA_LARGE_BATCH      = 8
JA_LARGE_GRAD_ACCUM = 8
JA_LARGE_LR         = 3e-4
JA_LARGE_EARLY_STOP = 999  # effectively disabled

if os.path.exists(f"{JA_LARGE_OUT}/final/config.json"):
    print(f"[skip] checkpoint exists at {JA_LARGE_OUT}/final")
else:
    train_byt5(
        model_name="google/byt5-large",
        output_dir=JA_LARGE_OUT,
        target_langs=["ja"],
        epochs=JA_LARGE_EPOCHS,
        batch_size=JA_LARGE_BATCH,
        grad_accum=JA_LARGE_GRAD_ACCUM,
        lr=JA_LARGE_LR,
        early_stop_patience=JA_LARGE_EARLY_STOP,
        save_total_limit=1,
        gradient_checkpointing=True,
    )

In [ ]:
# Hybrid inference v2: byt5-base multilingual for 15 langs, byt5-LARGE for ko/ja.

from pathlib import Path
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from datasets import load_dataset

HYBRID_LARGE_SAVE = "outputs/submission_dev_byt5_hybrid_large"
MULTI_CKPT = "./checkpoints/byt5-base/final"
LANG_CKPT = {
    "ko": "./checkpoints/byt5-ko-large/final",
    "ja": "./checkpoints/byt5-ja-large/final",
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

byt5_data = load_dataset("weerayut/multilexnorm2026-dev-pub")
test_df = byt5_data["test"].to_pandas()
test_df["pred"] = [list() for _ in range(len(test_df))]

def predict_rows(ckpt, indices, batch_size=128, max_input=64, max_target=64):
    if not indices:
        return
    flat_w, flat_l, sent_lens = [], [], []
    for i in indices:
        row = test_df.iloc[i]
        flat_w.extend(row["raw"])
        flat_l.extend([row["lang"]] * len(row["raw"]))
        sent_lens.append(len(row["raw"]))
    print(f"[{ckpt}] {len(flat_w)} words from {len(indices)} sentences")
    tok = AutoTokenizer.from_pretrained(ckpt)
    model = T5ForConditionalGeneration.from_pretrained(ckpt).to(device).eval()
    preds = []
    for s in range(0, len(flat_w), batch_size):
        cw, cl = flat_w[s:s + batch_size], flat_l[s:s + batch_size]
        enc = tok([f"{l}: {w}" for l, w in zip(cl, cw)],
                  return_tensors="pt", padding=True, truncation=True,
                  max_length=max_input).to(device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_target, num_beams=1)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    cursor = 0
    for n, i in zip(sent_lens, indices):
        test_df.at[i, "pred"] = preds[cursor:cursor + n]
        cursor += n
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

default_idx = [i for i in range(len(test_df)) if test_df.at[i, "lang"] not in LANG_CKPT]
lang_idx = {lang: [i for i in range(len(test_df)) if test_df.at[i, "lang"] == lang]
            for lang in LANG_CKPT}

print(f"default model -> {len(default_idx)} sentences")
for lang, idxs in lang_idx.items():
    print(f"{lang} model    -> {len(idxs)} sentences")

predict_rows(MULTI_CKPT, default_idx, batch_size=128)
# byt5-large: smaller inference batch to keep within VRAM
for lang in sorted(lang_idx):
    predict_rows(LANG_CKPT[lang], lang_idx[lang], batch_size=64)

Path(HYBRID_LARGE_SAVE).mkdir(parents=True, exist_ok=True)
test_df.to_json(f"{HYBRID_LARGE_SAVE}/predictions.json", orient="records")
test_df[["raw", "pred", "lang"]].head(3)

In [ ]:
from utils import zip_files_flat
zip_files_flat(HYBRID_LARGE_SAVE, f"{HYBRID_LARGE_SAVE}.zip")

## ByT5 Base ABLATION — ko=3.0 oversample (negative result)

Ablation study testing whether oversampling the lowest-resource language (Korean
= 2.05% of train words) helps. We weight ko 3× during training and keep all
other 16 languages at weight 1.0; everything else is identical to the main
training cell (cell 12).

**Result: this approach hurts**. On codabench test:

| metric | cell-12 (no oversample, MAIN) | this cell (ko=3.0) | Δ |
| --- | --- | --- | --- |
| err-weighted | 50.84 | 48.66 | -2.18 |
| ko per-lang  | 11.54 | -5.77 | -17.31 |
| ja per-lang  | 7.91  | 5.59  | -2.32 |

**Why oversampling hurt**: ko has only 13k training word pairs. Visiting the
same 13k pairs 3× per epoch causes the model to overfit to specific ko examples
without learning generalisable patterns. We confirmed this by checking the copy
rate on test ko: both the 15-language byt5-base (which never saw ko in training)
and the ko=3.0 oversampled model output the raw input unchanged for 97.1% of
ko test words — so the extra ko exposure produced essentially zero learning signal.

This negative result is preserved as a cell to document why the main training
runs **without** oversampling.

In [ ]:
# Ablation: train byt5-base unified WITH ko=3.0 oversample (negative result).
# This is the rejected variant; cell 12 (no oversample) is the WINNING config.

import os

ABLATION_OUT       = "./checkpoints/byt5-base-unified"
ABLATION_EPOCHS    = 5
ABLATION_BATCH     = 64
ABLATION_OVERSAMPLE = {"ko": 3.0}   # all other 16 langs default to 1.0

if os.path.exists(f"{ABLATION_OUT}/final/config.json"):
    print(f"[skip] checkpoint exists at {ABLATION_OUT}/final")
else:
    train_byt5(
        model_name="google/byt5-base",
        output_dir=ABLATION_OUT,
        epochs=ABLATION_EPOCHS,
        batch_size=ABLATION_BATCH,
        oversample_weights=ABLATION_OVERSAMPLE,
        num_workers=0,
        save_total_limit=1,
    )
